# Notebook 1 — Setup y exploración del dataset

**TP Visión por Computadora II — CEIA FIUBA**

Sistema de monitoreo de EPP (Equipamiento de Protección Personal) en obras de construcción.

Objetivos de este notebook:
1. Verificar que las dependencias estén correctamente instaladas
2. Descargar el dataset Construction-PPE desde Roboflow
3. Realizar un análisis exploratorio (EDA): distribución de clases, tamaños de bounding boxes y visualización de muestras
4. Exportar la configuración del dataset para su uso en los notebooks siguientes

In [ ]:
# Detectar si se está ejecutando en Google Colab
import importlib
EN_COLAB = importlib.util.find_spec("google.colab") is not None

if EN_COLAB:
    print("Entorno: Google Colab")
    print("Instalando dependencias...")
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "ultralytics", "roboflow"])

    # Clonar el repo y moverse a la raíz del proyecto
    import os
    if not os.path.exists("vision_computadora_2"):
        subprocess.check_call(["git", "clone", "-b", "feat/notebook-02-training",
                               "https://github.com/spardo83/vision_computadora_2.git"])
    os.chdir("vision_computadora_2")
    print(f"Directorio de trabajo: {os.getcwd()}")
else:
    print("Entorno: local")

## 1. Verificación del entorno

Se verifican las versiones de Python, PyTorch y Ultralytics instaladas.
En caso de no contar con GPU, el proceso se ejecuta en CPU (más lento para el entrenamiento, pero funcionalmente equivalente).

In [37]:
import sys
import platform
import torch
import ultralytics

print(f"Python:      {sys.version}")
print(f"Sistema:     {platform.system()} {platform.machine()}")
print(f"PyTorch:     {torch.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")

tiene_gpu = torch.cuda.is_available()
print(f"\nDispositivo: {'CUDA (' + torch.cuda.get_device_name(0) + ')' if tiene_gpu else 'CPU'}")

if not tiene_gpu:
    print("No hay GPU — el entrenamiento va a ser mas lento, pero funciona.")

Python:      3.11.13 (main, Oct  7 2025, 16:08:14) [Clang 20.1.4 ]
Sistema:     Darwin arm64
PyTorch:     2.11.0
Ultralytics: 8.4.33

Dispositivo: CPU
No hay GPU — el entrenamiento va a ser mas lento, pero funciona.


## 2. Descarga del dataset

Se utiliza el dataset **Construction Site Safety** de Roboflow Universe, que contiene 10 clases relacionadas con EPP en obras de construcción:

- EPP **presente**: `Hardhat`, `Mask`, `Safety Vest`
- EPP **ausente**: `NO-Hardhat`, `NO-Mask`, `NO-Safety Vest`
- Otros: `Person`, `Safety Cone`, `machinery`, `vehicle`

Para la descarga se requiere una API key gratuita de Roboflow, que se puede obtener en https://roboflow.com → Account → API Keys.

- **En entorno local:** la key se lee del archivo `.env` en la raíz del proyecto.
- **En Google Colab:** la key se configura en el panel de Secrets (icono de llave en el panel izquierdo) con el nombre `ROBOFLOW_API_KEY`.

In [38]:
import os
from pathlib import Path

if EN_COLAB:
    # En Colab se usa el sistema de Secrets (panel izquierdo > llave)
    # para no exponer la key en el código
    from google.colab import userdata
    api_key = userdata.get("ROBOFLOW_API_KEY")
    carpeta_datos = Path("data")
else:
    # En local se carga desde el archivo .env en la raíz del proyecto
    from dotenv import load_dotenv
    load_dotenv(Path("../.env"))
    api_key = os.environ.get("ROBOFLOW_API_KEY", "")
    carpeta_datos = Path("../data")

if not api_key:
    raise ValueError(
        "Falta la API key de Roboflow. "
        "En Colab: agregarla en Secrets con el nombre ROBOFLOW_API_KEY. "
        "En local: configurarla en el archivo .env en la raiz del proyecto."
    )

carpeta_datos.mkdir(exist_ok=True)
print(f"Directorio de datos: {carpeta_datos.resolve()}")

Directorio de datos: /Users/mferrari/git/fiuba/vision_computadora_2/data


In [39]:
from roboflow import Roboflow

rf = Roboflow(api_key=api_key)

# Referencia del dataset: https://universe.roboflow.com/roboflow-universe-projects/construction-site-safety
proyecto = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
version = proyecto.version(28)

# Se descarga en formato YOLOv8 (imagen + archivo .txt con las anotaciones)
ruta_descarga = str(carpeta_datos / "construction-ppe")
dataset_descargado = version.download("yolov8", location=ruta_descarga)

print(f"\nDescargado en: {dataset_descargado.location}")

loading Roboflow workspace...
loading Roboflow project...

Descargado en: /Users/mferrari/git/fiuba/vision_computadora_2/data/construction-ppe


In [40]:
import yaml

ruta_dataset = Path(dataset_descargado.location)

# Cantidad de imágenes por split (train/valid/test)
for nombre_split in ["train", "valid", "test"]:
    carpeta_imagenes = ruta_dataset / nombre_split / "images"
    if carpeta_imagenes.exists():
        cantidad = len(list(carpeta_imagenes.glob("*.jpg"))) + len(list(carpeta_imagenes.glob("*.png")))
        print(f"  {nombre_split:6s}: {cantidad:5d} imagenes")

# data.yaml es el archivo de configuración que YOLO necesita para entrenar.
# Contiene la lista de clases y las rutas a cada split.
with open(ruta_dataset / "data.yaml") as f:
    config_yaml = yaml.safe_load(f)

nombres_clases = config_yaml["names"]
cantidad_clases = config_yaml["nc"]

print(f"\n{cantidad_clases} clases: {nombres_clases}")

  train :  2605 imagenes
  valid :   114 imagenes
  test  :    82 imagenes

10 clases: ['Hardhat', 'Mask', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest', 'Person', 'Safety Cone', 'Safety Vest', 'machinery', 'vehicle']


## 3. Análisis exploratorio (EDA)

Antes de entrenar cualquier modelo, es fundamental comprender la composición del dataset.
Como se vio en la clase 2, un buen análisis exploratorio permite identificar desbalances de clases,
anomalías en las anotaciones y características relevantes de las imágenes.

### 3.1 Distribución de clases

Se analiza la cantidad de instancias por clase en cada split para evaluar si existe desbalance.
Un desbalance significativo puede afectar negativamente el rendimiento del modelo en las clases minoritarias.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

plt.rcParams["figure.dpi"] = 120
sns.set_theme(style="whitegrid", palette="husl")


def contar_instancias_por_clase(carpeta_labels: Path, nombres_clases: list) -> Counter:
    """
    Lee todos los archivos .txt de labels YOLO en una carpeta
    y cuenta cuantas veces aparece cada clase.
    
    Cada linea de un archivo .txt tiene el formato:
        <id_clase> <centro_x> <centro_y> <ancho> <alto>
    Solo nos interesa el id_clase (primer valor) para contar.
    """
    conteo = Counter()
    for archivo_label in carpeta_labels.glob("*.txt"):
        with open(archivo_label) as f:
            for linea in f:
                id_clase = int(linea.strip().split()[0])
                conteo[nombres_clases[id_clase]] += 1
    return conteo


# Contar instancias en cada split
conteo_por_split = {}
for nombre_split in ["train", "valid", "test"]:
    carpeta_labels = ruta_dataset / nombre_split / "labels"
    if carpeta_labels.exists():
        conteo_por_split[nombre_split] = contar_instancias_por_clase(carpeta_labels, nombres_clases)

# Graficar
paleta_colores = sns.color_palette("husl", len(nombres_clases))
fig, ejes = plt.subplots(1, len(conteo_por_split), figsize=(18, 5))
if len(conteo_por_split) == 1:
    ejes = [ejes]

for eje, (nombre_split, conteo) in zip(ejes, conteo_por_split.items()):
    # Ordenar de mayor a menor para que se lea mejor
    conteo_ordenado = dict(sorted(conteo.items(), key=lambda x: x[1], reverse=True))
    barras = eje.barh(list(conteo_ordenado.keys()), list(conteo_ordenado.values()), color=paleta_colores)
    eje.set_title(f"{nombre_split} ({sum(conteo.values())} instancias)", fontsize=13, fontweight="bold")
    eje.set_xlabel("Cantidad de instancias")
    # Agregar el numero al lado de cada barra
    for barra, valor in zip(barras, conteo_ordenado.values()):
        eje.text(valor + 20, barra.get_y() + barra.get_height() / 2, str(valor), va="center", fontsize=9)

plt.suptitle("Distribucion de clases por split", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(carpeta_datos / "class_distribution.png", bbox_inches="tight", dpi=150)
plt.show()

**Observaciones:**

- La clase `Person` es la más frecuente en todos los splits (~9600 en train), lo cual es esperable ya que cada trabajador en la imagen genera una anotación de persona además de las anotaciones de EPP.
- Existe un desbalance entre las clases de EPP: por ejemplo, `Mask` tiene aproximadamente la mitad de instancias que `Safety Vest` o `NO-Safety Vest`. Esto podría afectar la capacidad del modelo para detectar mascarillas con la misma precisión que los otros elementos.
- Las clases `vehicle` y `Mask` son las menos representadas en el set de entrenamiento (~1500 instancias cada una).
- La proporción relativa entre clases se mantiene razonablemente consistente entre los tres splits, lo cual indica que la partición fue estratificada.

### 3.2 Tamaños de los bounding boxes

Se analiza la distribución de ancho, alto y área de los bounding boxes en el set de entrenamiento.
En formato YOLO, estos valores están normalizados entre 0 y 1 (relativos al tamaño de la imagen).

Los objetos con área muy reducida (< 0.01) suelen ser más difíciles de detectar para el modelo.

In [ ]:
def extraer_tamanios_bboxes(carpeta_labels: Path):
    """
    Lee los archivos de labels y extrae ancho, alto y area de cada bbox.
    Los valores estan normalizados (0 a 1), porque asi los guarda YOLO.
    """
    anchos, altos, areas = [], [], []
    for archivo_label in carpeta_labels.glob("*.txt"):
        with open(archivo_label) as f:
            for linea in f:
                partes = linea.strip().split()
                if len(partes) >= 5:
                    ancho_bbox = float(partes[3])
                    alto_bbox = float(partes[4])
                    anchos.append(ancho_bbox)
                    altos.append(alto_bbox)
                    areas.append(ancho_bbox * alto_bbox)
    return anchos, altos, areas


carpeta_labels_train = ruta_dataset / "train" / "labels"
anchos, altos, areas = extraer_tamanios_bboxes(carpeta_labels_train)

fig, ejes = plt.subplots(1, 3, figsize=(15, 4))

ejes[0].hist(anchos, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
ejes[0].set_title("Ancho (normalizado)")
ejes[0].set_xlabel("Ancho relativo")
ejes[0].set_ylabel("Frecuencia")

ejes[1].hist(altos, bins=50, color="coral", edgecolor="white", alpha=0.8)
ejes[1].set_title("Alto (normalizado)")
ejes[1].set_xlabel("Alto relativo")

ejes[2].hist(areas, bins=50, color="mediumseagreen", edgecolor="white", alpha=0.8)
ejes[2].set_title("Area (ancho x alto)")
ejes[2].set_xlabel("Area relativa")

plt.suptitle(f"Tamanios de bounding boxes — train ({len(anchos):,} instancias)", fontsize=13)
plt.tight_layout()
plt.savefig(carpeta_datos / "bbox_stats.png", bbox_inches="tight", dpi=150)
plt.show()

print(f"\nEstadisticas del area:")
print(f"  Mediana: {np.median(areas):.4f}")
print(f"  Media:   {np.mean(areas):.4f}")
print(f"  Min:     {np.min(areas):.4f}")
print(f"  Max:     {np.max(areas):.4f}")

**Observaciones:**

- La distribución de tamaños está fuertemente sesgada hacia valores pequeños: la mediana del área es ~0.018, mientras que la media es ~0.060. Esto indica que la mayoría de los objetos anotados ocupan menos del 2% de la superficie total de la imagen.
- El sesgo es esperable en imágenes de obras de construcción, donde la cámara suele estar a distancia considerable de los trabajadores y sus elementos de protección.
- Los objetos muy pequeños (área < 0.01) representan una proporción significativa del dataset. Esto tiene implicancias directas en la configuración del entrenamiento: resoluciones de entrada mayores (e.g., 640×640 en lugar de 320×320) permiten al modelo capturar mejor los detalles de estos objetos.
- El rango de áreas es amplio (de ~0.0001 a ~0.53), lo que significa que el modelo deberá manejar simultáneamente objetos muy pequeños (EPP a distancia) y objetos grandes (vehículos o maquinaria en primer plano). Las arquitecturas YOLO abordan esto mediante detección multi-escala con Feature Pyramid Networks (FPN).

### 3.3 Visualización de imágenes con anotaciones

Se seleccionan imágenes al azar del set de entrenamiento y se dibujan los bounding boxes
correspondientes. Se utiliza verde para indicar EPP presente y rojo para EPP ausente.

In [ ]:
import cv2
import random

# Colores por clase: verde = tiene EPP, rojo = le falta EPP, otros en colores neutros
COLOR_POR_CLASE = {
    "Hardhat":        (0, 200, 0),
    "Mask":           (0, 180, 0),
    "Safety Vest":    (0, 160, 0),
    "NO-Hardhat":     (220, 0, 0),
    "NO-Mask":        (200, 0, 0),
    "NO-Safety Vest": (180, 0, 0),
    "Person":         (100, 100, 255),
    "Safety Cone":    (255, 165, 0),
    "machinery":      (128, 0, 128),
    "vehicle":        (64, 64, 64),
}


def dibujar_anotaciones_yolo(ruta_imagen: Path, ruta_label: Path, nombres_clases: list) -> np.ndarray:
    """
    Lee una imagen y su archivo de labels YOLO, y dibuja los bounding boxes.
    
    El formato YOLO guarda las coordenadas normalizadas:
        <id_clase> <centro_x> <centro_y> <ancho> <alto>
    Hay que desnormalizarlas multiplicando por el ancho/alto real de la imagen
    para poder dibujar los rectangulos.
    """
    imagen = cv2.imread(str(ruta_imagen))
    imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
    alto_img, ancho_img = imagen.shape[:2]

    if ruta_label.exists():
        with open(ruta_label) as f:
            for linea in f:
                partes = linea.strip().split()
                id_clase = int(partes[0])
                centro_x = float(partes[1])
                centro_y = float(partes[2])
                ancho_bbox = float(partes[3])
                alto_bbox = float(partes[4])

                # Convertir de formato YOLO (centro + tamaño normalizado)
                # a coordenadas de esquinas en pixeles
                x1 = int((centro_x - ancho_bbox / 2) * ancho_img)
                y1 = int((centro_y - alto_bbox / 2) * alto_img)
                x2 = int((centro_x + ancho_bbox / 2) * ancho_img)
                y2 = int((centro_y + alto_bbox / 2) * alto_img)

                nombre_clase = nombres_clases[id_clase]
                color = COLOR_POR_CLASE.get(nombre_clase, (200, 200, 200))
                cv2.rectangle(imagen, (x1, y1), (x2, y2), color, 2)
                cv2.putText(imagen, nombre_clase, (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return imagen


# Agarrar 8 imagenes al azar del train
todas_las_imagenes_train = list((ruta_dataset / "train" / "images").glob("*.jpg"))
random.seed(42)
imagenes_muestra = random.sample(todas_las_imagenes_train, min(8, len(todas_las_imagenes_train)))

fig, ejes = plt.subplots(2, 4, figsize=(20, 10))
ejes = ejes.flatten()

for i, ruta_imagen in enumerate(imagenes_muestra):
    ruta_label = ruta_dataset / "train" / "labels" / (ruta_imagen.stem + ".txt")
    imagen_anotada = dibujar_anotaciones_yolo(ruta_imagen, ruta_label, nombres_clases)
    ejes[i].imshow(imagen_anotada)
    ejes[i].axis("off")
    ejes[i].set_title(ruta_imagen.name[:30], fontsize=8)

plt.suptitle("Muestras del dataset (verde = tiene EPP, rojo = le falta EPP)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(carpeta_datos / "sample_annotations.png", bbox_inches="tight", dpi=150)
plt.show()

**Observaciones:**

- Los bounding boxes coinciden correctamente con los objetos en las imágenes, lo cual confirma la calidad de las anotaciones del dataset.
- Se observa que algunas imágenes son mosaicos (composiciones de 4 imágenes en una). Esto es producto de *mosaic augmentation*, una técnica de data augmentation que ya viene aplicada en el dataset. Esta técnica obliga al modelo a detectar objetos en distintas escalas, posiciones y contextos simultáneamente.
- Los objetos de EPP (cascos, chalecos, mascarillas) tienden a ser pequeños en relación al tamaño total de la imagen, consistente con lo observado en el análisis de bounding boxes de la sección anterior.
- Se verifica visualmente la convención de colores: verde para EPP presente (`Hardhat`, `Mask`, `Safety Vest`) y rojo para EPP ausente (`NO-Hardhat`, `NO-Mask`, `NO-Safety Vest`), lo cual facilita la interpretación de las detecciones.

## 4. Resumen y configuración para los siguientes notebooks

Se genera una tabla resumen del dataset y se exporta la configuración (rutas y clases)
a un archivo JSON que será utilizado por los notebooks 2, 3 y 4.

In [44]:
import pandas as pd

filas_resumen = []
for nombre_split in ["train", "valid", "test"]:
    carpeta_imagenes = ruta_dataset / nombre_split / "images"
    carpeta_labels = ruta_dataset / nombre_split / "labels"
    if carpeta_imagenes.exists():
        num_imagenes = len(list(carpeta_imagenes.glob("*.jpg"))) + len(list(carpeta_imagenes.glob("*.png")))
        conteo = contar_instancias_por_clase(carpeta_labels, nombres_clases)
        total_anotaciones = sum(conteo.values())
        filas_resumen.append({
            "Split": nombre_split,
            "Imagenes": num_imagenes,
            "Anotaciones": total_anotaciones,
            "Anotaciones/imagen": round(total_anotaciones / num_imagenes, 1),
        })

tabla_resumen = pd.DataFrame(filas_resumen)
print("=== Resumen del dataset ===")
print(tabla_resumen.to_string(index=False))

=== Resumen del dataset ===
Split  Imagenes  Anotaciones  Anotaciones/imagen
train      2605        36895                14.2
valid       114          697                 6.1
 test        82          760                 9.3


In [ ]:
import json

ruta_yaml_dataset = str(ruta_dataset / "data.yaml")

configuracion_dataset = {
    "dataset_yaml": ruta_yaml_dataset,
    "num_classes": cantidad_clases,
    "class_names": nombres_clases,
}

with open(carpeta_datos / "dataset_config.json", "w") as f:
    json.dump(configuracion_dataset, f, indent=2)

print("Configuracion guardada en dataset_config.json")
print(f"Ruta al YAML del dataset: {ruta_yaml_dataset}")
print(f"\nEste archivo sera leido por los notebooks 2, 3 y 4.")